# Advanced Problems with Solutions: Python 3.9 Features

This notebook contains advanced practice problems based on selected Python 3.9 features:

- `zoneinfo` for time zone aware datetime handling
- `math.gcd` with multiple arguments
- `math.lcm`
- dictionary union operators `|` and `|=`
- `str.removeprefix()` and `str.removesuffix()`

Each problem includes a realistic task, a reference solution, tests, and best-practice notes.

> Requires Python 3.9 or newer.

In [1]:
import sys
import math
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

assert sys.version_info >= (3, 9), "This notebook requires Python 3.9+"

## Problem 1 — Convert UTC Log Records to Local Time

You receive log records where timestamps are stored in UTC as ISO strings ending in `Z`.

Example:

```python
"2024-03-31T00:30:00Z"
```

Write `convert_utc_log_time(timestamp, target_tz)`.

Rules:

- accept only strings ending in `Z`;
- parse them as UTC datetimes;
- convert them to a target IANA time zone such as `"Europe/Sofia"`;
- return a timezone-aware `datetime`;
- raise `ValueError` for malformed timestamps;
- raise `ZoneInfoNotFoundError` for invalid time zone names.

In [2]:
def convert_utc_log_time(timestamp: str, target_tz: str) -> datetime:
    """Convert a UTC ISO-8601 log timestamp ending in Z to a target time zone."""
    if not isinstance(timestamp, str) or not timestamp.endswith("Z"):
        raise ValueError("Timestamp must be a string ending in 'Z'")

    try:
        naive = datetime.fromisoformat(timestamp.removesuffix("Z"))
    except ValueError as exc:
        raise ValueError(f"Malformed timestamp: {timestamp!r}") from exc

    if naive.tzinfo is not None:
        raise ValueError("Timestamp should not contain an explicit offset when it ends in Z")

    aware_utc = naive.replace(tzinfo=timezone.utc)
    return aware_utc.astimezone(ZoneInfo(target_tz))

In [3]:
dt = convert_utc_log_time("2024-03-31T00:30:00Z", "Europe/Sofia")
assert dt.tzinfo is not None
assert dt.hour == 2
assert dt.minute == 30
assert str(dt.tzinfo) == "Europe/Sofia"

try:
    convert_utc_log_time("2024-03-31T00:30:00", "Europe/Sofia")
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for timestamp without Z")

try:
    convert_utc_log_time("not-a-dateZ", "Europe/Sofia")
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for malformed timestamp")

print("Problem 1 tests passed.")

Problem 1 tests passed.


### Solution Notes

The input timestamp is explicitly UTC, so using `.replace(tzinfo=timezone.utc)` is appropriate. Do not use `.replace()` to convert between time zones; use `.astimezone()` for conversion.

## Problem 2 — Schedule a Global Meeting Safely

Write `meeting_times(local_start, source_tz, participant_timezones)`.

The function receives a naive local datetime representing the meeting time in the source time zone.

It should return a dictionary mapping each participant time zone to the corresponding local meeting time.

Rules:

- `local_start` must be naive;
- attach the source time zone using `ZoneInfo`;
- convert using `.astimezone()`;
- include the source time zone in the result;
- reject duplicate participant time zones;
- preserve insertion order.

In [4]:
def meeting_times(
    local_start: datetime,
    source_tz: str,
    participant_timezones: list[str],
) -> dict[str, datetime]:
    """Return meeting start times in multiple IANA time zones."""
    if local_start.tzinfo is not None:
        raise ValueError("local_start must be naive")

    all_timezones = [source_tz, *participant_timezones]

    if len(all_timezones) != len(set(all_timezones)):
        raise ValueError("Duplicate time zones are not allowed")

    aware_source = local_start.replace(tzinfo=ZoneInfo(source_tz))

    return {
        tz_name: aware_source.astimezone(ZoneInfo(tz_name))
        for tz_name in all_timezones
    }

In [5]:
result = meeting_times(
    datetime(2024, 7, 1, 10, 0),
    "Europe/Sofia",
    ["UTC", "America/New_York", "Asia/Tokyo"],
)

assert list(result) == ["Europe/Sofia", "UTC", "America/New_York", "Asia/Tokyo"]
assert result["Europe/Sofia"].hour == 10
assert result["UTC"].hour == 7
assert result["Asia/Tokyo"].hour == 16

try:
    meeting_times(datetime(2024, 7, 1, 10, 0, tzinfo=timezone.utc), "UTC", [])
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for aware local_start")

try:
    meeting_times(datetime(2024, 7, 1, 10, 0), "UTC", ["UTC"])
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for duplicate time zones")

print("Problem 2 tests passed.")

Problem 2 tests passed.


### Solution Notes

This problem emphasizes a common workflow: attach a known time zone to a naive datetime, then convert to other zones using `.astimezone()`.

## Problem 3 — Merge Configuration Layers with Dictionary Union

You are building configuration from several layers:

1. defaults,
2. environment overrides,
3. user overrides.

Write `merge_config(defaults, env, user)`.

Rules:

- use dictionary union `|`;
- later layers override earlier layers;
- do not mutate any input dictionary;
- validate that all keys are strings;
- return the merged dictionary.

In [6]:
def merge_config(
    defaults: dict[str, object],
    env: dict[str, object],
    user: dict[str, object],
) -> dict[str, object]:
    """Merge configuration dictionaries without mutating inputs."""
    for layer_name, layer in (("defaults", defaults), ("env", env), ("user", user)):
        if not all(isinstance(key, str) for key in layer):
            raise TypeError(f"All keys in {layer_name} must be strings")

    return defaults | env | user

In [7]:
defaults = {"host": "localhost", "port": 8000, "debug": False}
env = {"port": 9000}
user = {"debug": True}

merged = merge_config(defaults, env, user)

assert merged == {"host": "localhost", "port": 9000, "debug": True}
assert defaults == {"host": "localhost", "port": 8000, "debug": False}
assert env == {"port": 9000}
assert user == {"debug": True}

try:
    merge_config({"a": 1}, {2: "bad"}, {})
except TypeError:
    pass
else:
    raise AssertionError("Expected TypeError for non-string key")

print("Problem 3 tests passed.")

Problem 3 tests passed.


### Solution Notes

`d1 | d2` creates a new dictionary. When keys overlap, values from the right-hand dictionary win. This makes layered configuration readable and explicit.

## Problem 4 — Apply Runtime Overrides In Place

Write `apply_runtime_overrides(config, overrides)`.

Rules:

- use dictionary update union `|=`;
- mutate and return `config`;
- reject overrides containing keys not already present in `config`;
- this prevents accidental typo-based configuration keys.

In [8]:
def apply_runtime_overrides(
    config: dict[str, object],
    overrides: dict[str, object],
) -> dict[str, object]:
    """Apply overrides in place, rejecting unknown keys."""
    unknown = overrides.keys() - config.keys()
    if unknown:
        raise KeyError(f"Unknown config keys: {sorted(unknown)}")

    config |= overrides
    return config

In [9]:
config = {"debug": False, "timeout": 30}
same_object = apply_runtime_overrides(config, {"debug": True})

assert config == {"debug": True, "timeout": 30}
assert same_object is config

try:
    apply_runtime_overrides(config, {"timeuot": 10})
except KeyError as exc:
    assert "timeuot" in str(exc)
else:
    raise AssertionError("Expected KeyError for unknown key")

print("Problem 4 tests passed.")

Problem 4 tests passed.


### Solution Notes

`|=` mutates the dictionary on the left. This is useful for intentional in-place updates, but it should be used carefully when callers may still need the original dictionary.

## Problem 5 — Normalize Log Lines with `removeprefix()` and `removesuffix()`

You receive log lines with optional prefixes and suffixes:

```python
"(log) [2024-01-01T10:00:00] Started OK\n"
"(debug) [2024-01-01T10:00:01] Cache hit\n"
```

Write `clean_log_line(line)`.

Rules:

- remove exactly one of these prefixes if present: `"(log) "`, `"(debug) "`, `"(warn) "`;
- remove one trailing newline if present;
- do not use `strip`, `lstrip`, or `rstrip`;
- return the cleaned line.

In [10]:
def clean_log_line(line: str) -> str:
    """Clean a log line by removing known exact prefixes and one trailing newline."""
    if not isinstance(line, str):
        raise TypeError("line must be a string")

    for prefix in ("(log) ", "(debug) ", "(warn) "):
        cleaned = line.removeprefix(prefix)
        if cleaned != line:
            line = cleaned
            break

    return line.removesuffix("\n")

In [11]:
assert clean_log_line("(log) [2024-01-01T10:00:00] Started OK\n") == "[2024-01-01T10:00:00] Started OK"
assert clean_log_line("(debug) cache hit\n") == "cache hit"
assert clean_log_line("(warn) low disk") == "low disk"
assert clean_log_line("ordinary line\n") == "ordinary line"
assert clean_log_line("ordinary line") == "ordinary line"

try:
    clean_log_line(123)
except TypeError:
    pass
else:
    raise AssertionError("Expected TypeError for non-string input")

print("Problem 5 tests passed.")

Problem 5 tests passed.


### Solution Notes

`removeprefix()` and `removesuffix()` remove exact substrings. They are safer than `lstrip()` and `rstrip()` when you want to remove a specific token rather than a set of characters.

## Problem 6 — Simplify Many Fractions with Multi-Argument `math.gcd`

Write `normalize_integer_ratio(values)`.

Given a sequence of integers, divide all values by their common greatest divisor.

Examples:

```python
[12, 18, 30] -> [2, 3, 5]
[-12, 18, 30] -> [-2, 3, 5]
```

Rules:

- use `math.gcd` with multiple arguments;
- reject an empty input;
- reject an all-zero input;
- preserve signs.

In [12]:
def normalize_integer_ratio(values: list[int]) -> list[int]:
    """Divide a list of integers by their greatest common divisor."""
    if not values:
        raise ValueError("values cannot be empty")

    if not all(isinstance(value, int) for value in values):
        raise TypeError("all values must be integers")

    divisor = math.gcd(*values)

    if divisor == 0:
        raise ValueError("cannot normalize an all-zero ratio")

    return [value // divisor for value in values]

In [13]:
assert normalize_integer_ratio([12, 18, 30]) == [2, 3, 5]
assert normalize_integer_ratio([-12, 18, 30]) == [-2, 3, 5]
assert normalize_integer_ratio([7]) == [1]
assert normalize_integer_ratio([0, 10, 20]) == [0, 1, 2]

try:
    normalize_integer_ratio([])
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for empty input")

try:
    normalize_integer_ratio([0, 0, 0])
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for all-zero input")

print("Problem 6 tests passed.")

Problem 6 tests passed.


### Solution Notes

Python 3.9 allows `math.gcd` to accept more than two arguments. This makes ratio normalization simpler and avoids manual loops or `functools.reduce`.

## Problem 7 — Find a Shared Schedule Interval with `math.lcm`

Several recurring tasks repeat every N minutes.

Write `next_common_interval(periods)` that returns the number of minutes until all tasks align again.

Rules:

- use `math.lcm`;
- reject an empty input;
- reject zero or negative periods;
- reject non-integer periods.

In [14]:
def next_common_interval(periods: list[int]) -> int:
    """Return the least common multiple of positive integer periods."""
    if not periods:
        raise ValueError("periods cannot be empty")

    if not all(isinstance(period, int) for period in periods):
        raise TypeError("all periods must be integers")

    if not all(period > 0 for period in periods):
        raise ValueError("all periods must be positive")

    return math.lcm(*periods)

In [15]:
assert next_common_interval([2, 3, 4]) == 12
assert next_common_interval([15, 20, 30]) == 60
assert next_common_interval([7]) == 7

for bad in ([], [0, 10], [-5, 10]):
    try:
        next_common_interval(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Expected ValueError for {bad!r}")

try:
    next_common_interval([1.5, 3])
except TypeError:
    pass
else:
    raise AssertionError("Expected TypeError for non-integer periods")

print("Problem 7 tests passed.")

Problem 7 tests passed.


### Solution Notes

`math.lcm` was added in Python 3.9. It is ideal for synchronization problems such as recurring jobs, calendar intervals, and repeating cycles.

## Problem 8 — Build a Production-Style Log Ingestion Pipeline

Combine multiple Python 3.9 features.

You receive raw log records like this:

```python
"(log) 2024-01-01T10:00:00Z service=api status=200\n"
```

Write `parse_log_record(line, target_tz, defaults)`.

Rules:

- remove the exact `"(log) "` prefix;
- remove one trailing newline;
- parse the first token as a UTC timestamp ending in `Z`;
- convert it to `target_tz` using `zoneinfo`;
- parse remaining `key=value` tokens into a dictionary;
- merge parsed fields over `defaults` using dictionary union;
- include the converted datetime under key `"timestamp"`;
- raise `ValueError` for malformed tokens.

In [16]:
def parse_log_record(
    line: str,
    target_tz: str,
    defaults: dict[str, object],
) -> dict[str, object]:
    """Parse and normalize a production-style log record."""
    cleaned = line.removeprefix("(log) ").removesuffix("\n")
    parts = cleaned.split()

    if not parts:
        raise ValueError("empty log line")

    timestamp_text, *tokens = parts
    timestamp = convert_utc_log_time(timestamp_text, target_tz)

    parsed: dict[str, object] = {}

    for token in tokens:
        if "=" not in token:
            raise ValueError(f"Malformed token: {token!r}")

        key, value = token.split("=", maxsplit=1)

        if not key:
            raise ValueError(f"Empty key in token: {token!r}")

        parsed[key] = value

    return defaults | parsed | {"timestamp": timestamp}

In [17]:
record = parse_log_record(
    "(log) 2024-01-01T10:00:00Z service=api status=200\n",
    "Europe/Sofia",
    {"service": "unknown", "env": "prod"},
)

assert record["service"] == "api"
assert record["status"] == "200"
assert record["env"] == "prod"
assert record["timestamp"].tzinfo is not None
assert str(record["timestamp"].tzinfo) == "Europe/Sofia"

try:
    parse_log_record("(log) 2024-01-01T10:00:00Z service api\n", "UTC", {})
except ValueError as exc:
    assert "Malformed token" in str(exc)
else:
    raise AssertionError("Expected ValueError for malformed token")

try:
    parse_log_record("\n", "UTC", {})
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for empty log line")

print("Problem 8 tests passed.")

Problem 8 tests passed.


### Solution Notes

This problem combines exact string cleanup, time zone conversion, and dictionary union. This is the kind of integration task where Python 3.9 features become especially useful in real applications.

## Best-Practice Summary

- Prefer `zoneinfo.ZoneInfo` for IANA time zones in Python 3.9+.
- Keep UTC timestamps timezone-aware before converting them.
- Use `.astimezone()` for time zone conversion.
- Use `dict1 | dict2` when you want a new merged dictionary.
- Use `dict1 |= dict2` only when intentional mutation is desired.
- Use `removeprefix()` and `removesuffix()` for exact string removal.
- Do not replace exact prefix/suffix removal with `strip`, `lstrip`, or `rstrip`.
- Use multi-argument `math.gcd` and `math.lcm` to avoid unnecessary manual loops.
- Test both successful behavior and failure behavior.